# Calculus from scratch: derivatives, gradients and the chain rule

*Mathematics for Machine Learning from First Principles · Lesson 4*

Every training algorithm in this course asks the same question again and again: *if I nudge this parameter a little, how does the loss change?* Calculus answers it. In this notebook you will build that answer yourself in plain Python: you will estimate derivatives from function values, measure how accurate those estimates are, and use them to test derivatives that you work out by hand.

## Learning objectives

By the end of this notebook you will be able to:

1. approach a limit numerically, and recognise when a limit does not exist;
2. estimate a derivative with forward and central finite differences, and explain why the central difference is more accurate;
3. choose a step size $h$ by balancing truncation error against rounding error, using a sweep of $h$ as evidence;
4. compute partial derivatives and the gradient of a function of several variables;
5. apply the chain rule to a composite function and confirm the result numerically;
6. check an analytic gradient against a numerical one with a relative error and a sensible tolerance;
7. read an expression as a computational graph and compute its gradient with a backward pass.

## Dependencies

**Tier 0 — Python standard library only.** The core cells import only `math`, `random` and `sys`, which come with every Python installation. The notebook runs in JupyterLite in your browser and on the course JupyterHub server without installing anything.

One cell is optional. It is tagged `visualization` and draws a plot with Matplotlib (which uses NumPy); both libraries are available in JupyterLite and on the course server. Nothing later in the notebook depends on that cell, and the core cell above it prints a table with the same numbers, so you can skip it.

**How to use this notebook.** Run the cells in order from the top; *Run → Run All Cells* also works. The exercises near the end start as stubs that raise `NotImplementedError`, so the whole notebook runs cleanly before you attempt them. Each exercise is followed by a `check(...)` cell, which prints `not implemented yet` until you write your answer, confirms a correct answer, and stops with an `AssertionError` that names the failing case when an answer is wrong. Reference solutions are in the final section; attempt each exercise before you read them.

In [ ]:
import math
import random
import sys


def _matches(actual, expected, rel_tol, abs_tol):
    """Compare a result with an expected value; numbers are compared approximately."""
    if expected is None:
        return actual is None
    if isinstance(expected, bool):
        return isinstance(actual, bool) and actual == expected
    if isinstance(expected, (int, float)):
        return (
            isinstance(actual, (int, float))
            and not isinstance(actual, bool)
            and math.isclose(actual, expected, rel_tol=rel_tol, abs_tol=abs_tol)
        )
    if isinstance(expected, (list, tuple)):
        return (
            isinstance(actual, (list, tuple))
            and len(actual) == len(expected)
            and all(_matches(a, e, rel_tol, abs_tol) for a, e in zip(actual, expected))
        )
    return actual == expected


def check(name, fn, cases, rel_tol=1e-6, abs_tol=1e-9):
    """Test an exercise function and report the outcome.

    Each case is a tuple (description, args, expected), and the function is
    called as fn(*args). `expected` is either a value (numbers are compared
    with math.isclose, lists element by element) or a function that receives
    the result and returns True when the result is acceptable.

    Prints "<name>: not implemented yet" and returns False while fn raises
    NotImplementedError. Prints a pass line and returns True when every case
    passes. Raises AssertionError, naming the failing case, when a result is wrong.
    """
    for description, args, expected in cases:
        try:
            result = fn(*args)
        except NotImplementedError:
            print(f"{name}: not implemented yet")
            return False
        if callable(expected):
            try:
                passed = bool(expected(result))
            except Exception:  # the result does not have the expected shape
                passed = False
            wanted = "a result that matches the description"
        else:
            passed = _matches(result, expected, rel_tol, abs_tol)
            wanted = repr(expected)
        if not passed:
            hint = " Did you forget a return statement?" if result is None else ""
            raise AssertionError(f"{name}: {description}: got {result!r}, expected {wanted}.{hint}")
    count = len(cases)
    print(f"{name}: correct ({count} check{'' if count == 1 else 's'} passed)")
    return True


print(f"Python {sys.version.split()[0]}: the check() helper is ready.")

## Theory

### Functions and limits

A function $f$ maps an input $x$ to an output $f(x)$. The **limit** of $f$ at a point $a$ is the value that $f(x)$ approaches as $x$ gets closer and closer to $a$:

$$\lim_{x \to a} f(x) = L .$$

The function does not have to be defined at $a$ itself: $\sin(x)/x$ cannot be evaluated at $x = 0$, yet its values approach $1$. A limit exists only if the values from the left and from the right approach the *same* number. A computer cannot shrink the distance to $a$ all the way to zero, but it can evaluate $f(a - h)$ and $f(a + h)$ for $h = 10^{-1}, 10^{-2}, \dots$ and show where the values settle.

### The derivative

The derivative of $f$ at $x$ is the limit of the slope of a chord as the chord shrinks to a point:

$$f'(x) = \lim_{h \to 0} \frac{f(x + h) - f(x)}{h} .$$

It measures how fast $f$ changes near $x$: for a small change $\Delta x$, $f(x + \Delta x) \approx f(x) + f'(x)\,\Delta x$.

### Finite differences and their errors

Choosing a small but non-zero $h$ turns the definition into an estimate. Taylor's theorem says how far off it is:

$$\underbrace{\frac{f(x + h) - f(x)}{h}}_{\text{forward difference}} = f'(x) + \frac{h}{2} f''(\xi_1), \qquad \underbrace{\frac{f(x + h) - f(x - h)}{2h}}_{\text{central difference}} = f'(x) + \frac{h^2}{6} f'''(\xi_2),$$

for some points $\xi_1$ and $\xi_2$ close to $x$. This **truncation error** shrinks in proportion to $h$ for the forward difference and in proportion to $h^2$ for the central difference.

There is a second source of error. A Python `float` carries about 16 significant digits; the gap between $1$ and the next representable number is the **machine epsilon**, $\varepsilon \approx 2.2 \times 10^{-16}$. Each computed value of $f$ can be wrong by roughly $\varepsilon\,|f(x)|$, and the difference quotient divides that **rounding error** by $h$. Up to small constant factors, the total error is

$$\text{error}(h) \approx \underbrace{C\,h^{p}}_{\text{truncation}} + \underbrace{\frac{\varepsilon\,|f(x)|}{h}}_{\text{rounding}},$$

with $p = 1$ for the forward difference and $p = 2$ for the central difference. A smaller $h$ reduces the first term but inflates the second, so there is a best step: near $\sqrt{\varepsilon} \approx 10^{-8}$ for the forward difference and near $\varepsilon^{1/3} \approx 10^{-5}$ for the central difference.

### Partial derivatives and the gradient

A function of several variables, such as a loss of $n$ parameters, has one **partial derivative** per input. To find it, vary that input and hold all the others fixed:

$$\frac{\partial f}{\partial x_i}(\mathbf{x}) = \lim_{h \to 0} \frac{f(\mathbf{x} + h\,\mathbf{e}_i) - f(\mathbf{x})}{h},$$

where $\mathbf{e}_i$ is the vector with $1$ in position $i$ and $0$ everywhere else. Collecting the partial derivatives gives the **gradient**

$$\nabla f(\mathbf{x}) = \left( \frac{\partial f}{\partial x_1}(\mathbf{x}), \dots, \frac{\partial f}{\partial x_n}(\mathbf{x}) \right),$$

which points in the direction in which $f$ increases fastest; its length says how fast.

### The chain rule

If $y = f(u)$ and $u = g(x)$, then

$$\frac{dy}{dx} = \frac{dy}{du}\,\frac{du}{dx} = f'(g(x))\,g'(x) .$$

A model's loss is a long chain of simple operations, and the chain rule differentiates it one link at a time. When an input reaches the output along several paths, the contributions of the paths are added.

### Why this matters for machine learning

Training adjusts parameters $\theta$ to reduce a loss $L(\theta)$. Gradient descent, the subject of the next lesson, repeats the update

$$\theta_{t+1} = \theta_t - \eta\,\nabla L(\theta_t),$$

where $\eta > 0$ is the learning rate. Every step needs the gradient. Libraries compute it with the chain rule, a method known as backpropagation. Finite differences are far too slow for models with millions of parameters, but they are the standard way to **check** that a hand-written or library gradient is correct.

## Step-by-step implementation

Each step implements one idea in a code cell, tests it with `assert` statements in the next cell, and then states the behaviour you should see. If an assertion fails, its message names the quantity that went wrong.

### Step 1 — Approach a limit numerically

`approach(f, a, steps)` evaluates $f(a - h)$ and $f(a + h)$ for $h = 10^{-1}, 10^{-2}, \dots, 10^{-\text{steps}}$. We try it at $a = 0$ on three functions:

- $\sin(x)/x$, which is undefined at $0$ but has the limit $1$ there;
- $(1 + x)^{1/x}$, whose limit at $0$ is Euler's number $e = 2.718281828\ldots$;
- a step function that jumps from $0$ to $1$ at $x = 0$, so it has no limit there.

In [ ]:
def approach(f, a, steps=8):
    """Return rows (h, f(a - h), f(a + h)) for h = 10**-1, 10**-2, ..., 10**-steps."""
    rows = []
    for k in range(1, steps + 1):
        h = 10.0 ** -k
        rows.append((h, f(a - h), f(a + h)))
    return rows


def print_limit_table(title, rows):
    print(title)
    print(f"{'h':>7}  {'f(a - h)':>18}  {'f(a + h)':>18}")
    for h, left, right in rows:
        print(f"{h:>7.0e}  {left:>18.15f}  {right:>18.15f}")
    print()


def sinc(x):
    return math.sin(x) / x


def compound(x):
    return (1 + x) ** (1 / x)


def step(x):
    return 1.0 if x >= 0 else 0.0


print_limit_table("sin(x) / x near 0", approach(sinc, 0.0))
print_limit_table("(1 + x) ** (1 / x) near 0", approach(compound, 0.0, steps=15))
print_limit_table("step(x) near 0", approach(step, 0.0, steps=4))

In [ ]:
# Tests for Step 1
rows = approach(sinc, 0.0)
assert len(rows) == 8 and math.isclose(rows[0][0], 0.1), "approach() should start at h = 0.1 and take 8 steps"
gaps = [abs(right - 1.0) for _, _, right in rows]
assert all(later <= earlier for earlier, later in zip(gaps, gaps[1:])), "sin(x)/x should get closer to 1 as h shrinks"
assert math.isclose(rows[-1][1], 1.0) and math.isclose(rows[-1][2], 1.0), "sin(x)/x should approach 1 from both sides"

rows = approach(compound, 0.0, steps=15)
errors = [abs(right - math.e) for _, _, right in rows]
assert min(errors) < 1e-7, "(1 + x) ** (1 / x) should come within 1e-7 of e"
assert errors[-1] > 1000 * min(errors), "at h = 1e-15, rounding error should have spoilt the estimate"

rows = approach(step, 0.0, steps=4)
assert all(left == 0.0 and right == 1.0 for _, left, right in rows), "the step function should jump from 0 to 1"
print("Step 1 tests passed.")

**Expected behaviour.** For $\sin(x)/x$, both columns approach $1$, gaining about two correct digits per row, and from $h = 10^{-8}$ they print as exactly `1.000000000000000`. For $(1 + x)^{1/x}$, the values approach $e = 2.718281828\ldots$ until $h \approx 10^{-8}$ and then drift away again; at $h = 10^{-15}$ the right-hand value is about $3.04$, wrong in the first decimal place. That is rounding error, which Step 4 explains. The step function gives $0$ on the left and $1$ on the right however small $h$ becomes, so it has no limit at $0$. The test cell prints `Step 1 tests passed.`

### Step 2 — The forward difference

The forward difference replaces the limit in the definition of $f'(x)$ by a small, fixed $h$. To measure its truncation error, we estimate the derivative of $\sin$ at $x = 1$, where the exact answer is $\cos 1$, and divide each error by $h$. If the error is proportional to $h$, that ratio should settle near $|f''(1)|/2 = \sin(1)/2 \approx 0.42$.

In [ ]:
def forward_difference(f, x, h=1e-8):
    """Estimate f'(x) with the forward difference (f(x + h) - f(x)) / h."""
    return (f(x + h) - f(x)) / h


exact = math.cos(1.0)  # the derivative of sin at x = 1
print(f"{'h':>7}  {'estimate':>16}  {'error':>9}  {'error / h':>9}")
for k in range(1, 6):
    h = 10.0 ** -k
    estimate = forward_difference(math.sin, 1.0, h)
    error = abs(estimate - exact)
    print(f"{h:>7.0e}  {estimate:>16.12f}  {error:>9.2e}  {error / h:>9.4f}")

In [ ]:
# Tests for Step 2
def square(x):
    return x * x


# For f(x) = x**2 the forward difference is exactly 2x + h, so its error is h.
assert math.isclose(forward_difference(square, 3.0, 0.1), 6.1), "for x**2 at 3 with h = 0.1, expected 6.1"
for h in (1e-1, 1e-2, 1e-3):
    ratio = (forward_difference(square, 3.0, h) - 6.0) / h
    assert math.isclose(ratio, 1.0, rel_tol=1e-6), f"for x**2 the error should equal h, but error / h = {ratio}"
assert abs(forward_difference(math.sin, 1.0) - math.cos(1.0)) < 1e-7, "with h = 1e-8, sin'(1) should be accurate to about 1e-8"
print("Step 2 tests passed.")

**Expected behaviour.** Every tenfold reduction of $h$ makes the error about ten times smaller, and the last column settles at about $0.42 = \sin(1)/2$: the forward difference is *first-order* accurate. Even with $h = 10^{-5}$ the error is still about $4 \times 10^{-6}$. The test cell prints `Step 2 tests passed.`

### Step 3 — The central difference

The central difference evaluates $f$ on both sides of $x$. The first-order error terms of the two sides cancel, which leaves an error proportional to $h^2$. We repeat the experiment of Step 2 and divide each central-difference error by $h^2$; the ratio should settle near $|f'''(1)|/6 = \cos(1)/6 \approx 0.090$.

In [ ]:
def central_difference(f, x, h=1e-5):
    """Estimate f'(x) with the central difference (f(x + h) - f(x - h)) / (2h)."""
    return (f(x + h) - f(x - h)) / (2 * h)


print(f"{'h':>7}  {'forward error':>13}  {'central error':>13}  {'central error / h**2':>20}")
for k in range(1, 5):
    h = 10.0 ** -k
    forward_error = abs(forward_difference(math.sin, 1.0, h) - exact)
    central_error = abs(central_difference(math.sin, 1.0, h) - exact)
    print(f"{h:>7.0e}  {forward_error:>13.2e}  {central_error:>13.2e}  {central_error / h**2:>20.4f}")

In [ ]:
# Tests for Step 3
# A central difference is exact for a quadratic, apart from rounding error.
assert math.isclose(central_difference(square, 3.0, 0.1), 6.0, rel_tol=1e-12), "for x**2 at 3, expected exactly 6"
# For x**3 the central difference equals 3x**2 + h**2, so at x = 2 with h = 0.1 it gives 12.01.
assert math.isclose(central_difference(lambda x: x ** 3, 2.0, 0.1), 12.01, rel_tol=1e-10), "for x**3 at 2 with h = 0.1, expected 12.01"
for k in (2, 3):
    h = 10.0 ** -k
    ratio = abs(central_difference(math.sin, 1.0, h) - exact) / h**2
    assert math.isclose(ratio, math.cos(1.0) / 6, rel_tol=1e-3), f"error / h**2 should be close to cos(1)/6, got {ratio}"
assert abs(central_difference(math.sin, 1.0) - exact) < 1e-10, "with h = 1e-5, sin'(1) should be accurate to about 1e-11"
print("Step 3 tests passed.")

**Expected behaviour.** The central error falls about a hundredfold for every tenfold reduction of $h$ (second-order accuracy), and the last column settles at about $0.090 = \cos(1)/6$. At $h = 10^{-4}$ the central difference is already more than ten thousand times more accurate than the forward difference, for the same cost of two evaluations of $f$. The test cell prints `Step 3 tests passed.`

### Step 4 — The step-size trade-off

Steps 2 and 3 suggest that a smaller $h$ is always better. It is not. `error_sweep` measures both errors for $h = 10^{-1}, 10^{-2}, \dots, 10^{-15}$, again for $\sin'(1)$. Watch each error fall, reach a minimum, and then *rise* again as rounding error takes over.

In [ ]:
def error_sweep(f, x, exact_derivative, exponents=range(1, 16)):
    """Return rows (h, forward error, central error) for h = 10**-k, k in exponents."""
    rows = []
    for k in exponents:
        h = 10.0 ** -k
        forward_error = abs(forward_difference(f, x, h) - exact_derivative)
        central_error = abs(central_difference(f, x, h) - exact_derivative)
        rows.append((h, forward_error, central_error))
    return rows


sweep = error_sweep(math.sin, 1.0, math.cos(1.0))
print(f"{'h':>7}  {'forward error':>13}  {'central error':>13}")
for h, forward_error, central_error in sweep:
    print(f"{h:>7.0e}  {forward_error:>13.2e}  {central_error:>13.2e}")

best_forward = min(sweep, key=lambda row: row[1])
best_central = min(sweep, key=lambda row: row[2])
epsilon = sys.float_info.epsilon
print(f"\nmachine epsilon: {epsilon:.3e}")
print(f"best forward step: h = {best_forward[0]:.0e} (error {best_forward[1]:.1e}); "
      f"theory: about sqrt(eps) = {math.sqrt(epsilon):.1e}")
print(f"best central step: h = {best_central[0]:.0e} (error {best_central[2]:.1e}); "
      f"theory: about eps ** (1/3) = {epsilon ** (1 / 3):.1e}")

In [ ]:
# Tests for Step 4
def exponent(h):
    """The k in h = 10**-k."""
    return round(-math.log10(h))


assert exponent(best_forward[0]) in (7, 8, 9), f"the forward difference should do best near h = 1e-8, not {best_forward[0]:.0e}"
assert exponent(best_central[0]) in (4, 5, 6), f"the central difference should do best near h = 1e-5, not {best_central[0]:.0e}"
assert best_central[2] < best_forward[1] / 100, "at its best step, the central difference should be far more accurate"
smallest = sweep[-1]
assert smallest[1] > 1000 * best_forward[1] and smallest[2] > 1000 * best_central[2], "at h = 1e-15, rounding error should dominate"
print("Step 4 tests passed.")

**Expected behaviour.** Both columns first fall: the forward error by a factor of about $10$ per row and the central error by a factor of about $100$. The forward error is smallest at $h = 10^{-8}$ (about $10^{-8}$) and the central error at $h = 10^{-5}$ (about $10^{-11}$); on some computers the best step is a neighbouring power of ten. Below those steps, both errors grow again, irregularly but on average by almost a factor of $10$ per row, and at $h = 10^{-15}$ both are around $10^{-2}$. Rounding error is the reason: below the best step, it grows faster than truncation error shrinks. A good default is $h = 10^{-5}$ for central differences, which is why `central_difference` uses it. The test cell prints `Step 4 tests passed.`

**Optional visualisation.** The next cell is tagged `visualization` and plots the sweep on logarithmic axes with Matplotlib. It is optional: the table above contains the same numbers, and nothing later uses this cell. On a log–log plot, the falling part of each curve is a straight line whose slope is the order of the method (1 for the forward difference, 2 for the central difference), and the rising part has a slope of about $-1$.

In [ ]:
# Optional: needs Matplotlib, which JupyterLite and the course server provide.
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("Matplotlib is not available here; the table above shows the same results.")

if plt is not None:
    hs = [row[0] for row in sweep]
    tiny = 1e-17  # a logarithmic axis cannot show an error of exactly zero
    fig, ax = plt.subplots(figsize=(6.5, 4))
    ax.loglog(hs, [max(row[1], tiny) for row in sweep], marker="o", label="forward difference")
    ax.loglog(hs, [max(row[2], tiny) for row in sweep], marker="s", label="central difference")
    ax.set_xlabel("step size h")
    ax.set_ylabel("absolute error in the estimate of sin'(1)")
    ax.set_title("Truncation error falls with h; rounding error rises")
    ax.legend()
    plt.show()

### Step 5 — Partial derivatives

From now on, a function of several variables takes a single list, `point`, in the same way that a model's loss takes a list of parameters. `partial_derivative(f, point, i)` applies the central difference to coordinate `i` only. It must not change the caller's list, so it works on copies.

Our example is

$$f(x, y) = x^2 y + \sin y, \qquad \frac{\partial f}{\partial x} = 2xy, \qquad \frac{\partial f}{\partial y} = x^2 + \cos y .$$

To find $\partial f / \partial x$, treat $y$ as a constant; to find $\partial f / \partial y$, treat $x$ as a constant.

In [ ]:
def partial_derivative(f, point, i, h=1e-5):
    """Central-difference estimate of the partial derivative of f along coordinate i."""
    ahead = list(point)   # copies, so that the caller's list is left unchanged
    behind = list(point)
    ahead[i] += h
    behind[i] -= h
    return (f(ahead) - f(behind)) / (2 * h)


def f_xy(point):
    x, y = point
    return x * x * y + math.sin(y)


p = [2.0, 1.0]
print(f"df/dx at {p}: numerical {partial_derivative(f_xy, p, 0):.10f}, exact {2 * 2.0 * 1.0:.10f}")
print(f"df/dy at {p}: numerical {partial_derivative(f_xy, p, 1):.10f}, exact {2.0 * 2.0 + math.cos(1.0):.10f}")

In [ ]:
# Tests for Step 5
p = [2.0, 1.0]
assert math.isclose(partial_derivative(f_xy, p, 0), 4.0, rel_tol=1e-9), "df/dx at (2, 1) should be 2 * 2 * 1 = 4"
assert math.isclose(partial_derivative(f_xy, p, 1), 4.0 + math.cos(1.0), rel_tol=1e-9), "df/dy at (2, 1) should be 4 + cos(1)"
assert p == [2.0, 1.0], "partial_derivative must not modify the point it is given"
# Along a coordinate that f ignores, the partial derivative is zero.
assert math.isclose(partial_derivative(lambda q: q[0] * q[0], [3.0, 5.0], 1), 0.0, abs_tol=1e-12), "x**2 does not depend on y"
print("Step 5 tests passed.")

**Expected behaviour.** Both numerical partial derivatives agree with the exact values to all ten printed decimals: $\partial f/\partial x = 4$ and $\partial f/\partial y = 4 + \cos 1 \approx 4.5403023059$. The list `p` is unchanged afterwards. The test cell prints `Step 5 tests passed.`

### Step 6 — The chain rule, checked numerically

Consider a single artificial neuron with a logistic (sigmoid) activation and a squared-error loss. For a fixed input $x$, bias $b$ and target $y$, the loss as a function of the weight $w$ is a chain of three links:

$$z = w x + b, \qquad a = \sigma(z) = \frac{1}{1 + e^{-z}}, \qquad L = (a - y)^2 .$$

Each link has a simple local derivative: $dz/dw = x$, $da/dz = \sigma(z)\,(1 - \sigma(z))$ and $dL/da = 2(a - y)$. The chain rule multiplies them:

$$\frac{dL}{dw} = \frac{dL}{da}\,\frac{da}{dz}\,\frac{dz}{dw} = 2\,(a - y)\;\sigma(z)\,\bigl(1 - \sigma(z)\bigr)\;x .$$

We code this formula and compare it with a central difference at several weights.

In [ ]:
X_IN, BIAS, TARGET = 1.5, -0.5, 1.0   # a fixed input, bias and target


def sigmoid(z):
    return 1.0 / (1.0 + math.exp(-z))


def neuron_loss(w):
    a = sigmoid(w * X_IN + BIAS)
    return (a - TARGET) ** 2


def neuron_loss_derivative(w):
    """dL/dw by the chain rule: dL/da * da/dz * dz/dw."""
    z = w * X_IN + BIAS
    a = sigmoid(z)
    dL_da = 2 * (a - TARGET)
    da_dz = a * (1 - a)
    dz_dw = X_IN
    return dL_da * da_dz * dz_dw


print(f"{'w':>5}  {'chain rule':>14}  {'central difference':>18}  {'difference':>10}")
for w in (-2.0, -0.5, 0.0, 0.8, 2.5):
    analytic = neuron_loss_derivative(w)
    numerical = central_difference(neuron_loss, w)
    print(f"{w:>5.1f}  {analytic:>14.10f}  {numerical:>18.10f}  {abs(analytic - numerical):>10.1e}")

In [ ]:
# Tests for Step 6
for w in (-2.0, -0.5, 0.0, 0.8, 2.5):
    analytic = neuron_loss_derivative(w)
    numerical = central_difference(neuron_loss, w)
    assert math.isclose(analytic, numerical, rel_tol=1e-7, abs_tol=1e-10), f"chain rule and central difference disagree at w = {w}"
# A second composite function: the derivative of sin(x**2) is cos(x**2) * 2x.
for x in (-1.3, 0.4, 2.0):
    numerical = central_difference(lambda t: math.sin(t * t), x)
    assert math.isclose(math.cos(x * x) * 2 * x, numerical, rel_tol=1e-7), f"d/dx sin(x**2) is wrong at x = {x}"
# Forgetting the inner derivative dz/dw = x is a classic mistake, and the numerical check exposes it.
forgot_inner = neuron_loss_derivative(0.8) / X_IN
assert not math.isclose(forgot_inner, central_difference(neuron_loss, 0.8), rel_tol=1e-3), "dropping dz/dw should change the answer"
print("Step 6 tests passed.")

**Expected behaviour.** The chain-rule and central-difference columns agree to about ten decimal places, and the difference column shows values of about $10^{-11}$ or smaller. The derivative is negative for every weight in the table: the target is $1$, so increasing $w$ moves the output towards it and lowers the loss. The test cell prints `Step 6 tests passed.`

### Step 7 — The gradient

The gradient is the list of all partial derivatives. A central-difference gradient costs $2n$ evaluations of $f$ for $n$ inputs. That is fine for checking a few parameters, but hopeless for training a model with millions of them, which is why training uses analytic gradients. We compare the two for $f(x, y) = x^2 y + \sin y$ from Step 5.

In [ ]:
def numerical_gradient(f, point, h=1e-5):
    """Central-difference gradient of f at point: one partial derivative per coordinate."""
    return [partial_derivative(f, point, i, h) for i in range(len(point))]


def grad_f_xy(point):
    """Analytic gradient of f(x, y) = x**2 * y + sin(y)."""
    x, y = point
    return [2 * x * y, x * x + math.cos(y)]


for point in ([2.0, 1.0], [-1.0, 0.5], [0.3, -2.0]):
    analytic = grad_f_xy(point)
    numerical = numerical_gradient(f_xy, point)
    print(f"at {point}: analytic [{analytic[0]:.8f}, {analytic[1]:.8f}]"
          f"   numerical [{numerical[0]:.8f}, {numerical[1]:.8f}]")

In [ ]:
# Tests for Step 7
for point in ([2.0, 1.0], [-1.0, 0.5], [0.3, -2.0]):
    analytic = grad_f_xy(point)
    numerical = numerical_gradient(f_xy, point)
    assert len(numerical) == 2, "a function of two variables has a gradient with two components"
    for i in range(2):
        assert math.isclose(numerical[i], analytic[i], rel_tol=1e-8, abs_tol=1e-10), f"component {i} disagrees at {point}"
# One component per coordinate, in any number of dimensions.
assert len(numerical_gradient(lambda q: sum(t * t for t in q), [1.0, 2.0, 3.0, 4.0])) == 4
print("Step 7 tests passed.")

**Expected behaviour.** At each of the three points, the analytic and numerical gradients agree to all eight printed decimals; for example, at $(2, 1)$ both are $[4, 4.54030231]$. The test cell prints `Step 7 tests passed.`

### Step 8 — Gradient checking with a relative error

A fixed tolerance on each component does not suit every scale: an error of $10^{-6}$ is negligible in a gradient of size $10^{6}$ and enormous in one of size $10^{-6}$. A **relative error** compares the size of the disagreement with the size of the vectors themselves:

$$\text{relative error}(\mathbf{a}, \mathbf{b}) = \frac{\lVert \mathbf{a} - \mathbf{b} \rVert}{\lVert \mathbf{a} \rVert + \lVert \mathbf{b} \rVert},$$

where $\lVert \mathbf{v} \rVert = \sqrt{v_1^2 + \dots + v_n^2}$ is the Euclidean length. It is $0$ for identical vectors and never more than $1$; we define it as $0$ when both vectors are exactly zero.

**Choosing a tolerance.** With central differences and $h = 10^{-5}$ in double precision, the correct gradient of a smooth function usually gives a relative error between about $10^{-11}$ and $10^{-8}$, while a genuine bug, such as a missing factor or a wrong sign, usually gives $10^{-3}$ or more. A tolerance of $10^{-6}$ sits comfortably between the two. Check at a few random points rather than at "nice" ones such as $0$ or $1$, where some bugs vanish: $x$ and $x^2$ are equal at $x = 1$.

In [ ]:
TOLERANCE = 1e-6


def vector_norm(v):
    """Euclidean length of a list of numbers."""
    return math.sqrt(sum(component * component for component in v))


def relative_error(a, b):
    """||a - b|| / (||a|| + ||b||), defined as 0.0 when both vectors are zero."""
    difference = vector_norm([x - y for x, y in zip(a, b, strict=True)])
    scale = vector_norm(a) + vector_norm(b)
    return 0.0 if scale == 0.0 else difference / scale


def grad_f_xy_buggy(point):
    """A plausible mistake: the factor 2 from the derivative of x**2 is missing."""
    x, y = point
    return [x * y, x * x + math.cos(y)]


rng = random.Random(0)  # a fixed seed, so every run checks the same points
test_points = [[rng.uniform(-2, 2), rng.uniform(-2, 2)] for _ in range(5)]

print(f"{'point':>18}  {'correct gradient':>16}  {'buggy gradient':>14}")
for point in test_points:
    numerical = numerical_gradient(f_xy, point)
    good = relative_error(grad_f_xy(point), numerical)
    bad = relative_error(grad_f_xy_buggy(point), numerical)
    print(f"({point[0]:6.3f}, {point[1]:6.3f})  {good:>16.1e}  {bad:>14.1e}")

In [ ]:
# Tests for Step 8
assert relative_error([1.0, 2.0], [1.0, 2.0]) == 0.0, "identical vectors have relative error 0"
assert math.isclose(relative_error([3.0, 4.0], [-3.0, -4.0]), 1.0), "opposite vectors have relative error 1"
assert relative_error([0.0, 0.0], [0.0, 0.0]) == 0.0, "two zero vectors count as equal"
assert math.isclose(relative_error([1e6], [1e6 + 1.0]), relative_error([1.0], [1.0 + 1e-6])), "the measure should not depend on scale"
for point in test_points:
    numerical = numerical_gradient(f_xy, point)
    assert relative_error(grad_f_xy(point), numerical) < TOLERANCE, f"the correct gradient should pass at {point}"
    assert relative_error(grad_f_xy_buggy(point), numerical) > TOLERANCE, f"the buggy gradient should fail at {point}"
print("Step 8 tests passed.")

**Expected behaviour.** In the table, the correct gradient gives relative errors below $10^{-11}$, far below the tolerance of $10^{-6}$, while the gradient with the missing factor gives errors between about $0.01$ and $0.2$. Its smallest errors occur where $x$ is close to $0$, because there the missing factor in $2xy$ hardly matters; that is why a check should use several points. The test cell prints `Step 8 tests passed.`

### Step 9 — A computational graph and a backward pass

Software that trains models does not differentiate a whole formula symbolically. It records the computation as a **computational graph** of simple operations and then applies the chain rule backwards through that graph. Take

$$f(x, y) = (x y + \sin x)^2 .$$

Naming every intermediate result turns the expression into this graph, in which each node knows the derivatives of its output with respect to its inputs:

| node | operation | inputs | local derivatives |
| --- | --- | --- | --- |
| $a$ | $x \cdot y$ | $x$, $y$ | $\partial a/\partial x = y$, $\;\partial a/\partial y = x$ |
| $b$ | $\sin x$ | $x$ | $\partial b/\partial x = \cos x$ |
| $c$ | $a + b$ | $a$, $b$ | $\partial c/\partial a = 1$, $\;\partial c/\partial b = 1$ |
| $f$ | $c^2$ | $c$ | $\partial f/\partial c = 2c$ |

The **forward pass** computes $a$, $b$, $c$ and $f$ in that order and keeps every value. The **backward pass** starts from $\partial f/\partial f = 1$ and visits the nodes in reverse, multiplying by each local derivative. The number it stores at each node, $\partial f/\partial(\text{node})$, is that node's gradient. Because $x$ feeds both $a$ and $b$, its gradient is the **sum** of the contributions along the two paths:

$$\frac{\partial f}{\partial x} = \frac{\partial f}{\partial a}\,\frac{\partial a}{\partial x} + \frac{\partial f}{\partial b}\,\frac{\partial b}{\partial x} = 2c\,y + 2c\,\cos x .$$

In [ ]:
def graph_function(point):
    """f(x, y) = (x*y + sin(x))**2, evaluated directly."""
    x, y = point
    c = x * y + math.sin(x)
    return c * c


def forward_backward(x, y):
    """Forward and backward pass through the graph of f(x, y) = (x*y + sin(x))**2.

    Returns (f, [df/dx, df/dy], nodes), where nodes lists
    (name, operation, value, gradient) for every node of the graph.
    """
    # Forward pass: compute each node from its inputs and keep the values.
    a = x * y
    b = math.sin(x)
    c = a + b
    f = c * c

    # Backward pass: start at the output and apply the chain rule in reverse order.
    df_df = 1.0
    df_dc = df_df * 2 * c                      # f = c**2, so df/dc = 2c
    df_da = df_dc * 1.0                        # c = a + b, so dc/da = 1
    df_db = df_dc * 1.0                        # ... and dc/db = 1
    df_dx = df_da * y + df_db * math.cos(x)    # x reaches f through a and through b: add both paths
    df_dy = df_da * x                          # y reaches f only through a

    nodes = [
        ("x", "input", x, df_dx),
        ("y", "input", y, df_dy),
        ("a", "x * y", a, df_da),
        ("b", "sin(x)", b, df_db),
        ("c", "a + b", c, df_dc),
        ("f", "c ** 2", f, df_df),
    ]
    return f, [df_dx, df_dy], nodes


value, gradient, nodes = forward_backward(2.0, 3.0)
print(f"{'node':>4}  {'operation':>9}  {'value':>12}  {'df/d(node)':>12}")
for name, operation, node_value, node_gradient in nodes:
    print(f"{name:>4}  {operation:>9}  {node_value:>12.6f}  {node_gradient:>12.6f}")

In [ ]:
# Tests for Step 9
for x, y in ((2.0, 3.0), (-1.2, 0.7), (0.5, -2.5)):
    value, gradient, _ = forward_backward(x, y)
    assert math.isclose(value, graph_function([x, y])), "the forward pass should reproduce f(x, y)"
    error = relative_error(gradient, numerical_gradient(graph_function, [x, y]))
    assert error < TOLERANCE, f"backward pass and numerical gradient disagree at ({x}, {y}): relative error {error:.1e}"
# Both paths from x matter: keeping only the path through a fails the gradient check.
value, gradient, nodes = forward_backward(2.0, 3.0)
c = 2.0 * 3.0 + math.sin(2.0)
one_path_only = [2 * c * 3.0, gradient[1]]
assert relative_error(one_path_only, numerical_gradient(graph_function, [2.0, 3.0])) > TOLERANCE
print("Step 9 tests passed.")

**Expected behaviour.** At $(x, y) = (2, 3)$ the table shows $a = 6$, $b = \sin 2 \approx 0.909297$, $c \approx 6.909297$ and $f \approx 47.738$. The gradient column shows $\partial f/\partial c = 2c \approx 13.818595$, which passes unchanged to $a$ and $b$, then $\partial f/\partial x \approx 35.705$ and $\partial f/\partial y \approx 27.637$. The backward pass agrees with the numerical gradient, and dropping the path through $b$ fails the check. One forward pass and one backward pass produce *every* partial derivative, whatever the number of inputs; this is reverse-mode automatic differentiation, the idea behind backpropagation. The test cell prints `Step 9 tests passed.`

## Exercises

Each exercise gives you a function stub. Replace the line `raise NotImplementedError(...)` with your code, run the cell, then run the `check(...)` cell below it. Until you do, the check prints, for example, `Exercise 1 (second_derivative): not implemented yet`, and the rest of the notebook keeps working. You may call any function defined in the steps above.

### Exercise 1 — A central difference for the second derivative

The second derivative $f''(x)$ measures curvature: how quickly the slope itself changes. Applying the central-difference idea twice gives

$$f''(x) \approx \frac{f(x + h) - 2 f(x) + f(x - h)}{h^2},$$

whose truncation error is proportional to $h^2$. Because the numerator is divided by $h^2$ rather than $h$, rounding error grows faster as $h$ shrinks, so the best step is larger than for a first derivative; keep the default $h = 10^{-4}$.

Implement `second_derivative(f, x, h=1e-4)`. The check uses $x^3$ (whose second derivative is $6x$), $x^4$, $\sin$, $e^x$ and a straight line, and it calls your function both with and without an explicit `h`.

In [ ]:
def second_derivative(f, x, h=1e-4):
    """Estimate f''(x) with the central second difference."""
    # TODO: evaluate f at x - h, x and x + h, and combine the three values as in
    # the formula above. Use the h you are given.
    raise NotImplementedError("Exercise 1: implement second_derivative")

In [ ]:
SECOND_DERIVATIVE_CASES = [
    ("x**4 at x = 1 with h = 0.1 gives 12x**2 + 2h**2 = 12.02", (lambda x: x ** 4, 1.0, 0.1), 12.02),
    ("x**3 at x = 2 (6x = 12)", (lambda x: x ** 3, 2.0), 12.0),
    ("x**3 at x = -1 (6x = -6)", (lambda x: x ** 3, -1.0), -6.0),
    ("sin at pi/6 (-sin(pi/6) = -0.5)", (math.sin, math.pi / 6), -0.5),
    ("exp at 0 (exp(0) = 1)", (math.exp, 0.0), 1.0),
    ("a straight line has no curvature", (lambda x: 4 * x - 7, 3.0), 0.0),
]
exercise_1_passed = check("Exercise 1 (second_derivative)", second_derivative, SECOND_DERIVATIVE_CASES,
                          rel_tol=1e-6, abs_tol=1e-5)

### Exercise 2 — The gradient of a two-variable function

Work out by hand the gradient of

$$g(x, y) = e^{x} \cos y + x y^2 ,$$

then implement `grad_g(point)`, which returns `[dg/dx, dg/dy]` for `point = [x, y]`. Remember: for $\partial g/\partial x$ treat $y$ as a constant, and for $\partial g/\partial y$ treat $x$ as a constant. The check compares your formula with `numerical_gradient` at several points, which is exactly the gradient check of Step 8.

In [ ]:
def g(point):
    x, y = point
    return math.exp(x) * math.cos(y) + x * y * y


def grad_g(point):
    """Analytic gradient [dg/dx, dg/dy] of g(x, y) = exp(x) * cos(y) + x * y**2."""
    # TODO: unpack x, y = point and return a list holding the two partial derivatives
    # you derived by hand.
    raise NotImplementedError("Exercise 2: implement grad_g")

In [ ]:
GRAD_G_CASES = [
    (f"gradient of g at {point}", (point,), numerical_gradient(g, point))
    for point in ([0.0, 0.0], [1.0, 2.0], [-0.5, 1.5], [0.3, -2.2])
]
exercise_2_passed = check("Exercise 2 (grad_g)", grad_g, GRAD_G_CASES, rel_tol=1e-6, abs_tol=1e-8)

### Exercise 3 — A reusable gradient checker

Turn Step 8 into a function you can reuse in later lessons. `gradient_check(f, grad_f, points, h=1e-5, tol=1e-6)` should

1. compute the analytic gradient `grad_f(point)` and the numerical gradient `numerical_gradient(f, point, h)` at **every** point in `points`;
2. measure their `relative_error` at each point;
3. return a tuple `(passed, worst_error)`, where `worst_error` is the largest relative error found and `passed` is `worst_error <= tol`.

The check includes a gradient that is right at one point but wrong at another, a function whose gradient is about a million times larger than usual, and a point where the gradient is exactly zero. An absolute-error test, or a test of only the first point, will not pass.

In [ ]:
def gradient_check(f, grad_f, points, h=1e-5, tol=1e-6):
    """Compare grad_f with a central-difference gradient at every point.

    Returns (passed, worst_error).
    """
    # TODO: for each point, compare grad_f(point) with numerical_gradient(f, point, h)
    # using relative_error; keep the largest error and return (worst_error <= tol, worst_error).
    raise NotImplementedError("Exercise 3: implement gradient_check")

In [ ]:
def bowl(point):
    x, y = point
    return x * x + 3 * y * y


def grad_bowl(point):
    x, y = point
    return [2 * x, 6 * y]


def grad_bowl_wrong_y(point):
    """Wrong (6y**2 instead of 6y), but it agrees with the truth wherever y = 1."""
    x, y = point
    return [2 * x, 6 * y * y]


def big_bowl(point):
    return 1e6 * bowl(point)


def grad_big_bowl(point):
    return [1e6 * component for component in grad_bowl(point)]


def grad_bowl_slightly_off(point):
    """Every component 0.02% too large: a relative error of about 1e-4."""
    return [1.0002 * component for component in grad_bowl(point)]


GRADIENT_CHECK_CASES = [
    ("returns a pair (passed, worst_error)", (bowl, grad_bowl, [[1.0, 2.0]]),
     lambda result: len(result) == 2 and isinstance(result[0], bool)),
    ("the correct gradient of f_xy passes at the random test points", (f_xy, grad_f_xy, test_points),
     lambda result: result[0] is True and result[1] < 1e-8),
    ("the gradient with a missing factor 2 fails", (f_xy, grad_f_xy_buggy, test_points),
     lambda result: result[0] is False and result[1] > 1e-3),
    ("a gradient that is wrong only at the second point fails and reports that point's error (about 0.98)",
     (bowl, grad_bowl_wrong_y, [[0.5, 1.0], [1.5, -2.0]]), lambda result: result[0] is False and result[1] > 0.9),
    ("a correct gradient of size about 1e7 passes", (big_bowl, grad_big_bowl, [[1.5, -2.5], [-0.7, 0.9]]),
     lambda result: result[0] is True),
    ("a correct zero gradient at the minimum passes", (bowl, grad_bowl, [[0.0, 0.0]]),
     lambda result: result[0] is True and result[1] == 0.0),
    ("an error of about 1e-4 fails with the default tol = 1e-6", (bowl, grad_bowl_slightly_off, [[1.0, 1.0]]),
     lambda result: result[0] is False),
    ("... and passes with tol = 1e-3", (bowl, grad_bowl_slightly_off, [[1.0, 1.0]], 1e-5, 1e-3),
     lambda result: result[0] is True),
]
exercise_3_passed = check("Exercise 3 (gradient_check)", gradient_check, GRADIENT_CHECK_CASES)

### Exercise 4 (stretch) — Directional derivatives and the steepest direction

The **directional derivative** of $f$ at $\mathbf{x}$ along a direction $\mathbf{u}$ of length one is the rate at which $f$ changes as you move along $\mathbf{u}$:

$$D_{\mathbf{u}} f(\mathbf{x}) = \lim_{h \to 0} \frac{f(\mathbf{x} + h\,\mathbf{u}) - f(\mathbf{x} - h\,\mathbf{u})}{2h} = \nabla f(\mathbf{x}) \cdot \mathbf{u} .$$

Implement `directional_derivative(f, point, direction, h=1e-5)` with the central difference in the middle of that formula. First scale `direction` to length one by dividing each component by its `vector_norm`, because only the direction should matter, not its length. Do not modify `point` or `direction`.

The check confirms that moving along a coordinate axis gives the partial derivative, and that moving along the gradient itself gives the largest possible rate of increase, $\lVert \nabla f \rVert$. The opposite direction, $-\nabla f$, gives the fastest *decrease*, which is why gradient descent steps along $-\nabla L$.

In [ ]:
def directional_derivative(f, point, direction, h=1e-5):
    """Rate of change of f at point along direction (scaled to length one first)."""
    # TODO: 1. divide each component of direction by vector_norm(direction);
    #       2. build the points point + h*u and point - h*u as new lists;
    #       3. return the central difference of f between them.
    raise NotImplementedError("Exercise 4: implement directional_derivative")

In [ ]:
P = [2.0, 1.0]
G = grad_f_xy(P)                      # [4, 4 + cos(1)]
untouched_point, untouched_direction = [2.0, 1.0], [0.0, 5.0]
DIRECTIONAL_CASES = [
    ("along the x-axis: equals df/dx = 4", (f_xy, P, [1.0, 0.0]), G[0]),
    ("along (0, 5): the length must not matter, so it equals df/dy", (f_xy, P, [0.0, 5.0]), G[1]),
    ("along (3, 4): equals the dot product of the gradient with (0.6, 0.8)", (f_xy, P, [3.0, 4.0]),
     0.6 * G[0] + 0.8 * G[1]),
    ("along the gradient: equals its length", (f_xy, P, G), vector_norm(G)),
    ("against the gradient: equals minus its length (steepest descent)", (f_xy, P, [-G[0], -G[1]]), -vector_norm(G)),
    ("leaves point and direction unchanged", (f_xy, untouched_point, untouched_direction),
     lambda result: untouched_point == [2.0, 1.0] and untouched_direction == [0.0, 5.0]),
]
exercise_4_passed = check("Exercise 4 (directional_derivative)", directional_derivative, DIRECTIONAL_CASES,
                          rel_tol=1e-7, abs_tol=1e-9)

## Common errors and debugging guidance

- **The estimate gets worse as $h$ shrinks.** Rounding error is dominating. Use $h \approx 10^{-5}$ for central differences and $h \approx 10^{-8}$ for forward differences, and repeat the sweep of Step 4 if you are unsure. When $|x|$ is far from $1$, scale the step with it, for example $h = 10^{-5} \max(1, |x|)$.
- **A partial derivative depends on which coordinate you computed first.** The function modified the caller's list: `point[i] += h` changes the original. Copy it first with `list(point)`.
- **A gradient check passes, but training still misbehaves.** You may have checked only at a special point. Bugs such as writing $x$ instead of $x^2$, or $xy$ instead of $2xy$, vanish at $x = 0$ or $x = 1$. Check at several random points drawn from a seeded `random.Random`.
- **The relative error is large although the gradient looks right.** If both gradients are almost zero, for example at a minimum, the relative error compares two tiny, noisy numbers. Check away from stationary points, or look at the absolute error there.
- **The check fails only near certain points.** Finite differences assume a smooth function. At a kink, such as $|x|$ or $\max(0, x)$ at $0$, the central difference averages two different slopes. Choose check points away from kinks.
- **The chain rule is off by a factor.** A link is missing, most often the derivative of the inner function. When a variable is used more than once, as $x$ is in Step 9, add the contributions of every path.
- **Trigonometric results look wrong.** `math.sin` and `math.cos` work in radians, not degrees.
- **`assert a == b` fails for numbers that print the same.** Floating-point results are rarely exact; compare them with `math.isclose(a, b, rel_tol=..., abs_tol=...)`. A relative tolerance alone never accepts a non-zero result when the expected value is `0.0`, so give an `abs_tol` whenever the answer may be zero.

## Summary

- A limit is the value a function approaches. Numerically, evaluate on both sides with a shrinking $h$ and look for agreement.
- The forward difference has an error proportional to $h$ and the central difference an error proportional to $h^2$, for the same two function evaluations.
- Smaller is not always better: rounding error grows like $\varepsilon / h$, so the best step in double precision is about $10^{-8}$ for the forward difference and about $10^{-5}$ for the central difference.
- A partial derivative varies one input at a time; the gradient collects them and points in the direction of steepest increase.
- The chain rule multiplies local derivatives along a path and adds the contributions of different paths. A computational graph organises this as a forward pass followed by a backward pass, which is how backpropagation works.
- Gradient checking compares an analytic gradient with a central-difference one through the relative error $\lVert \mathbf{a} - \mathbf{b} \rVert / (\lVert \mathbf{a} \rVert + \lVert \mathbf{b} \rVert)$. Correct gradients usually give $10^{-8}$ or less, so a tolerance of $10^{-6}$ catches real bugs.

**Next:** Lesson 5 uses these gradients to minimise functions with gradient descent, $\theta_{t+1} = \theta_t - \eta\,\nabla L(\theta_t)$.

## Solutions

Reference solutions for the exercises follow. They have different names from the stubs, so running these cells never overwrites your own work. Each one is verified with the same `check` call and the same test cases as its exercise, which also proves that the tests accept a correct answer. Attempt each exercise before you read its solution.

In [ ]:
# Exercise 1: central second difference
def second_derivative_solution(f, x, h=1e-4):
    return (f(x + h) - 2 * f(x) + f(x - h)) / (h * h)


assert check("Exercise 1 (reference solution)", second_derivative_solution, SECOND_DERIVATIVE_CASES,
             rel_tol=1e-6, abs_tol=1e-5), "the reference solution must pass its own checks"

In [ ]:
# Exercise 2: dg/dx = exp(x) cos(y) + y**2 and dg/dy = -exp(x) sin(y) + 2xy
def grad_g_solution(point):
    x, y = point
    return [math.exp(x) * math.cos(y) + y * y, -math.exp(x) * math.sin(y) + 2 * x * y]


assert check("Exercise 2 (reference solution)", grad_g_solution, GRAD_G_CASES,
             rel_tol=1e-6, abs_tol=1e-8), "the reference solution must pass its own checks"

In [ ]:
# Exercise 3: the worst relative error over all points decides the verdict
def gradient_check_solution(f, grad_f, points, h=1e-5, tol=1e-6):
    worst_error = 0.0
    for point in points:
        error = relative_error(grad_f(point), numerical_gradient(f, point, h))
        worst_error = max(worst_error, error)
    return worst_error <= tol, worst_error


assert check("Exercise 3 (reference solution)", gradient_check_solution, GRADIENT_CHECK_CASES), \
    "the reference solution must pass its own checks"

In [ ]:
# Exercise 4: scale the direction to length one, then take a central difference along it
def directional_derivative_solution(f, point, direction, h=1e-5):
    length = vector_norm(direction)
    unit = [component / length for component in direction]
    ahead = [p + h * u for p, u in zip(point, unit)]
    behind = [p - h * u for p, u in zip(point, unit)]
    return (f(ahead) - f(behind)) / (2 * h)


assert check("Exercise 4 (reference solution)", directional_derivative_solution, DIRECTIONAL_CASES,
             rel_tol=1e-7, abs_tol=1e-9), "the reference solution must pass its own checks"